# Validate Silver MGB flood areas

## Purpose

Validate `02-silver.silver_mgb_flood_area` independently of the notebook that builds it. The validator compares the table with the selected Bronze snapshot row by row on `object_id`, rebuilds every usable geometry from its WKB, and checks the seven data-quality attributes: Consistency, Accuracy, Completeness, Auditability, Validity, Uniqueness, and Timeliness.

Evidence is persisted to `04-validation.silver_dq_results` **before** the STOP gate runs, so failed blocking evidence stays reviewable. The validator never changes the table.

## Cost

Bronze is read for `object_id`, the raw code, and lineage only, never for `geometry_json`. All checks come from one metrics row.

## 1. Declare expectations and read the published context

**What:** declare the source SRID, the allowed status and parse-method vocabularies, the degree screen, and the October 2026 reference counts once. Then summarize the published table, the newest terminal MGB load, and its latest Bronze validation.

**Reference counts are FLAG evidence only:** 63,684 MGB rows from the accepted Bronze evidence. They never filter or repair data.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
DECLARE OR REPLACE VARIABLE degree_screen_min_longitude DOUBLE DEFAULT 110.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_longitude DOUBLE DEFAULT 130.0;
DECLARE OR REPLACE VARIABLE degree_screen_min_latitude DOUBLE DEFAULT 0.0;
DECLARE OR REPLACE VARIABLE degree_screen_max_latitude DOUBLE DEFAULT 25.0;
DECLARE OR REPLACE VARIABLE reference_mgb_rows BIGINT DEFAULT 63684;
-- D-43 repair tolerance, declared here independently of the transformation
DECLARE OR REPLACE VARIABLE repair_area_tolerance_ratio DOUBLE DEFAULT 0.01;
DECLARE OR REPLACE VARIABLE repair_area_tolerance_sqm DOUBLE DEFAULT 10000;
DECLARE OR REPLACE VARIABLE source_area_srid INT DEFAULT 3857;
DECLARE OR REPLACE VARIABLE validation_rule_version STRING
    DEFAULT 'silver-mgb-flood-area-validation-v1';

CREATE OR REPLACE TEMPORARY VIEW mgb_flood_area_validation_context AS
WITH published AS (
    SELECT
        COUNT(*) AS output_rows,
        COUNT(DISTINCT run_id) AS run_count,
        MAX(run_id) AS run_id,
        COUNT(DISTINCT mgb_source_snapshot_id) AS published_snapshot_count,
        MAX(mgb_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(mgb_source_ingest_run_id) AS mgb_source_ingest_run_id,
        MAX(parse_rule_version) AS parse_rule_version
    FROM `02-silver`.silver_mgb_flood_area
),
latest_load AS (
    SELECT * EXCEPT (terminal_rank)
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
        ) AS terminal_rank
        FROM `01-bronze`.load_log
        WHERE table_name = 'flood_susceptibility'
          AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
    )
    WHERE terminal_rank = 1
),
bronze_validation AS (
    SELECT
        COUNT(*) AS bronze_validation_rows,
        COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) AS bronze_blocking_failures
    FROM (
        SELECT dq.*, DENSE_RANK() OVER (ORDER BY dq.run_ts DESC, dq.run_id DESC) AS validation_rank
        FROM `04-validation`.dq_results AS dq
        INNER JOIN latest_load AS load
          ON dq.table_name = load.table_name
         AND dq.snapshot_id = load.snapshot_id
    ) AS dq
    WHERE dq.validation_rank = 1
)
SELECT
    published.*,
    latest_load.status AS latest_load_status,
    latest_load.snapshot_id AS latest_mgb_snapshot_id,
    latest_load.rows_loaded AS latest_mgb_rows_loaded,
    bronze_validation.*
FROM published
CROSS JOIN latest_load
CROSS JOIN bronze_validation;

SELECT * FROM mgb_flood_area_validation_context;

## 2. Calculate one grouped metric row

**What:** join the published table to the selected Bronze snapshot on `object_id` in one full outer join, rebuild each usable geometry from WKB, and aggregate every metric in one pass.

**Why:** the full outer join finds Bronze rows that were never published and published rows with no Bronze source. Rebuilding usable geometry proves that the WKB that Tables 14 and 15 read is valid, nonempty, and in the source SRID.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mgb_flood_area_validation_metrics AS
WITH bronze AS (
    SELECT
        TRIM(mgb.object_id) AS object_id,
        NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), '') AS raw_susceptibility_code,
        TRY_CAST(mgb.source_shape_area AS DOUBLE) AS source_shape_area,
        mgb._ingest_run_id AS ingest_run_id
    FROM `01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN mgb_flood_area_validation_context AS context
    WHERE mgb._source_snapshot_id = context.latest_mgb_snapshot_id
),
published AS (
    SELECT
        area.*,
        CASE
            WHEN area.source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')
            THEN ST_SETSRID(ST_GEOMFROMWKB(area.geometry_wkb), source_geometry_srid)
        END AS usable_geometry
    FROM `02-silver`.silver_mgb_flood_area AS area
),
paired AS (
    SELECT bronze.object_id AS bronze_object_id, bronze.raw_susceptibility_code AS bronze_code,
           bronze.ingest_run_id AS bronze_ingest_run_id, bronze.source_shape_area, published.*,
           CASE
               WHEN published.source_geometry_status = 'REPAIRED_GEOMETRY'
               THEN ST_AREA(ST_TRANSFORM(published.usable_geometry, source_area_srid))
           END AS repaired_area_sqm
    FROM bronze
    FULL OUTER JOIN published
        ON bronze.object_id = published.object_id
),
counts AS (
    SELECT
        COUNT_IF(bronze_object_id IS NOT NULL) AS bronze_rows,
        COUNT_IF(bronze_object_id IS NOT NULL AND object_id IS NULL) AS bronze_rows_not_published,
        COUNT_IF(bronze_object_id IS NULL AND object_id IS NOT NULL) AS published_rows_without_bronze,
        COUNT_IF(bronze_object_id IS NOT NULL AND object_id IS NOT NULL
            AND NOT (bronze_code <=> raw_susceptibility_code)) AS raw_code_mismatches,
        COUNT_IF(object_id IS NOT NULL AND NOT (bronze_ingest_run_id <=> mgb_source_ingest_run_id))
            AS ingest_run_mismatches,
        COUNT_IF(object_id IS NOT NULL) - COUNT(DISTINCT object_id) AS duplicate_object_ids,
        COUNT_IF(object_id IS NOT NULL) - COUNT(DISTINCT mgb_flood_area_key) AS duplicate_keys,
        COUNT_IF(object_id IS NOT NULL AND mgb_flood_area_key <> SHA2(CONCAT_WS(
            '||', object_id, mgb_source_snapshot_id, parse_rule_version), 256)) AS key_mismatches,
        COUNT_IF(object_id IS NOT NULL AND (
            mgb_source_snapshot_id IS NULL OR mgb_source_ingest_run_id IS NULL
            OR parse_rule_version IS NULL OR run_id IS NULL OR source_load_ts IS NULL
        )) AS incomplete_lineage_rows,
        COUNT_IF(object_id IS NOT NULL AND source_geometry_status NOT IN (
            'BLANK_GEOMETRY', 'UNPARSEABLE_GEOMETRY', 'EMPTY_GEOMETRY',
            'UNSUPPORTED_GEOMETRY_TYPE', 'INVALID_GEOMETRY', 'REPAIRED_GEOMETRY', 'USABLE_GEOMETRY'
        )) AS unknown_status_rows,
        COUNT_IF(object_id IS NOT NULL AND geometry_parse_method NOT IN (
            'NOT_PARSED', 'GEOJSON_AS_PUBLISHED', 'GEOJSON_QUOTED_NUMBERS_NORMALIZED',
            'ESRI_JSON_CONVERTED', 'PARSE_FAILED'
        )) AS unknown_parse_method_rows,
        COUNT_IF(source_geometry_status IN ('BLANK_GEOMETRY', 'UNPARSEABLE_GEOMETRY')
            AND geometry_wkb IS NOT NULL) AS unparsed_rows_with_geometry,
        COUNT_IF(source_geometry_status IN ('EMPTY_GEOMETRY', 'UNSUPPORTED_GEOMETRY_TYPE',
            'INVALID_GEOMETRY', 'REPAIRED_GEOMETRY', 'USABLE_GEOMETRY') AND geometry_wkb IS NULL)
            AS parsed_rows_without_geometry,
        COUNT_IF(source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')) AS usable_rows,
        COUNT_IF(source_geometry_status = 'REPAIRED_GEOMETRY') AS repaired_rows,
        COUNT_IF(source_geometry_status = 'REPAIRED_GEOMETRY' AND NOT COALESCE(
            source_shape_area > 0
            AND ABS(repaired_area_sqm - source_shape_area)
                <= GREATEST(repair_area_tolerance_ratio * source_shape_area, repair_area_tolerance_sqm),
            FALSE
        )) AS repaired_rows_outside_tolerance,
        ROUND(COALESCE(SUM(CASE WHEN source_geometry_status = 'REPAIRED_GEOMETRY'
            THEN source_shape_area END), 0) / 1000000, 1) AS repaired_source_area_sqkm,
        COUNT_IF(source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY') AND (
            geometry_srid IS NULL OR geometry_srid <> source_geometry_srid
        )) AS usable_wrong_srid_rows,
        COUNT_IF(source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY') AND (
            usable_geometry IS NULL OR ST_ISEMPTY(usable_geometry) OR NOT ST_ISVALID(usable_geometry)
            OR REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(usable_geometry)), '^ST_', '')
                NOT IN ('POLYGON', 'MULTIPOLYGON')
        )) AS usable_rows_failing_rebuild,
        COUNT_IF(source_geometry_status IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY') AND (
            ST_XMIN(usable_geometry) < degree_screen_min_longitude
            OR ST_XMAX(usable_geometry) > degree_screen_max_longitude
            OR ST_YMIN(usable_geometry) < degree_screen_min_latitude
            OR ST_YMAX(usable_geometry) > degree_screen_max_latitude
        )) AS usable_outside_degree_screen_rows,
        COUNT_IF(object_id IS NOT NULL
            AND source_geometry_status NOT IN ('USABLE_GEOMETRY', 'REPAIRED_GEOMETRY')) AS unusable_rows,
        COUNT_IF(source_geometry_status = 'INVALID_GEOMETRY') AS invalid_geometry_rows,
        COUNT_IF(object_id IS NOT NULL AND raw_susceptibility_code IS NULL) AS blank_code_rows
    FROM paired
)
SELECT
    context.output_rows, context.run_count, context.published_snapshot_count,
    context.mgb_source_snapshot_id, context.latest_mgb_snapshot_id, context.latest_load_status,
    context.latest_mgb_rows_loaded, context.bronze_validation_rows, context.bronze_blocking_failures,
    counts.*
FROM counts
CROSS JOIN mgb_flood_area_validation_context AS context;

SELECT * FROM mgb_flood_area_validation_metrics;

## 3. Express STOP and FLAG checks across all seven attributes

**STOP** protects upstream safety and timeliness, complete one-to-one Bronze accounting, raw code fidelity, key and grain uniqueness, deterministic keys, complete lineage, known vocabularies, and geometry that Tables 14 and 15 can read safely.

**FLAG** keeps real source limitations visible with denominators: unusable and invalid geometry, polygons repaired under D-43, blank codes, geometry outside the degree screen, and differences from the historical reference count. A FLAG is not a failure.

**D-43 repair:** a STOP check recomputes the Web Mercator area of every `REPAIRED_GEOMETRY` row from its WKB and requires it to stay within 1% or 0.01 sq km of the MGB `Shape__Area`. The tolerance is declared in this notebook, independently of `08a`.

A failed blocking check is `FAIL`. A nonzero review finding is `FLAG`. Zero findings are `PASS`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW mgb_flood_area_validation_checks AS
WITH checks AS (
    -- one metrics row is unpivoted into check rows, so the source scan is not repeated per check
    SELECT INLINE(ARRAY(
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB load is successful and identified',
            'failed_rows', CAST(CASE WHEN latest_load_status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT') AND latest_mgb_rows_loaded > 0 THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Latest load status=', COALESCE(latest_load_status, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Latest MGB Bronze validation has no blocking failure',
            'failed_rows', CAST(CASE WHEN bronze_validation_rows > 0 THEN bronze_blocking_failures ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Bronze validation rows=', bronze_validation_rows, '; blocking failures=', bronze_blocking_failures)
        ),
        NAMED_STRUCT(
            'attribute', 'Timeliness', 'check_name', 'Published MGB snapshot is the latest validated MGB snapshot',
            'failed_rows', CAST(CASE WHEN published_snapshot_count = 1 AND mgb_source_snapshot_id <=> latest_mgb_snapshot_id THEN 0 ELSE 1 END AS BIGINT), 'total_rows', CAST(1 AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', COALESCE(mgb_source_snapshot_id, 'NULL'), '; latest=', COALESCE(latest_mgb_snapshot_id, 'NULL'))
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'One deterministic run represents the table',
            'failed_rows', CAST(ABS(run_count - 1) AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Distinct run IDs=', run_count)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Every row carries complete lineage',
            'failed_rows', CAST(incomplete_lineage_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Rows with incomplete lineage=', incomplete_lineage_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Auditability', 'check_name', 'Every row comes from the selected Bronze ingest run',
            'failed_rows', CAST(ingest_run_mismatches AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Ingest run mismatches=', ingest_run_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Every selected Bronze row is published',
            'failed_rows', CAST(bronze_rows_not_published AS BIGINT), 'total_rows', CAST(bronze_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Bronze rows not published=', bronze_rows_not_published, '; Bronze rows=', bronze_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Published rows equal the audited Bronze row count',
            'failed_rows', CAST(ABS(output_rows - latest_mgb_rows_loaded) AS BIGINT), 'total_rows', CAST(latest_mgb_rows_loaded AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published=', output_rows, '; audited rows_loaded=', latest_mgb_rows_loaded)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Unusable MGB geometry remains visible',
            'failed_rows', CAST(unusable_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Unusable rows=', unusable_rows, '; invalid geometry rows=', invalid_geometry_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Blank MGB codes remain visible',
            'failed_rows', CAST(blank_code_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Blank code rows=', blank_code_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Every published row has a Bronze source row',
            'failed_rows', CAST(published_rows_without_bronze AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Published rows without Bronze=', published_rows_without_bronze)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Raw susceptibility codes match Bronze',
            'failed_rows', CAST(raw_code_mismatches AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Raw code mismatches=', raw_code_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Consistency', 'check_name', 'Row count matches the historical reference',
            'failed_rows', CAST(ABS(output_rows - reference_mgb_rows) AS BIGINT), 'total_rows', CAST(reference_mgb_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Published=', output_rows, '; reference=', reference_mgb_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'MGB flood area keys are unique',
            'failed_rows', CAST(duplicate_keys AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate keys=', duplicate_keys)
        ),
        NAMED_STRUCT(
            'attribute', 'Uniqueness', 'check_name', 'One row exists per object_id, snapshot, and parse rule version',
            'failed_rows', CAST(duplicate_object_ids AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Duplicate object IDs=', duplicate_object_ids)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Keys are the deterministic hash of object, snapshot, and parse rule',
            'failed_rows', CAST(key_mismatches AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Key mismatches=', key_mismatches)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Geometry statuses and parse methods use the governed vocabulary',
            'failed_rows', CAST(unknown_status_rows + unknown_parse_method_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Unknown statuses=', unknown_status_rows, '; unknown parse methods=', unknown_parse_method_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Stored geometry presence matches the geometry status',
            'failed_rows', CAST(unparsed_rows_with_geometry + parsed_rows_without_geometry AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Unparsed rows with WKB=', unparsed_rows_with_geometry, '; parsed rows without WKB=', parsed_rows_without_geometry)
        ),
        NAMED_STRUCT(
            'attribute', 'Validity', 'check_name', 'Usable geometry is stored in the source SRID',
            'failed_rows', CAST(usable_wrong_srid_rows AS BIGINT), 'total_rows', CAST(usable_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Usable rows in another SRID=', usable_wrong_srid_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Usable geometry rebuilds as a valid nonempty polygon',
            'failed_rows', CAST(usable_rows_failing_rebuild AS BIGINT), 'total_rows', CAST(usable_rows AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Usable rows failing the WKB rebuild=', usable_rows_failing_rebuild)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Usable geometry lies inside the degree screen',
            'failed_rows', CAST(usable_outside_degree_screen_rows AS BIGINT), 'total_rows', CAST(usable_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Usable rows outside the degree screen=', usable_outside_degree_screen_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Accuracy', 'check_name', 'Repaired geometry keeps the MGB area within the D-43 tolerance',
            'failed_rows', CAST(repaired_rows_outside_tolerance AS BIGINT), 'total_rows', CAST(GREATEST(repaired_rows, 1) AS BIGINT),
            'action', 'stop',
            'details', CONCAT('Repaired rows outside 1% or 0.01 sq km of Shape__Area=', repaired_rows_outside_tolerance,
                '; repaired rows=', repaired_rows)
        ),
        NAMED_STRUCT(
            'attribute', 'Completeness', 'check_name', 'Repaired MGB geometry remains visible',
            'failed_rows', CAST(repaired_rows AS BIGINT), 'total_rows', CAST(output_rows AS BIGINT),
            'action', 'flag',
            'details', CONCAT('Polygons repaired under D-43=', repaired_rows,
                '; their MGB Shape__Area in Web Mercator sq km=', repaired_source_area_sqkm)
        )
    ))
    FROM mgb_flood_area_validation_metrics
),
context AS (
    SELECT
        run_id AS pipeline_run_id,
        mgb_source_snapshot_id AS source_snapshot_id,
        parse_rule_version AS rule_version,
        SHA2(CONCAT_WS('|', run_id, validation_rule_version), 256) AS validation_run_id,
        CURRENT_TIMESTAMP() AS run_ts
    FROM mgb_flood_area_validation_context
)
SELECT
    context.validation_run_id, context.pipeline_run_id, 'silver' AS layer,
    'silver_mgb_flood_area' AS table_name, context.source_snapshot_id, context.rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    checks.failed_rows, checks.total_rows,
    CASE WHEN checks.total_rows > 0
        THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action, checks.details, context.run_ts
FROM checks
CROSS JOIN context;

## 4. Persist evidence before enforcing the gate

**What:** `MERGE` the checks into the shared `04-validation.silver_dq_results` table, then enforce the STOP gate from the persisted rows.

**Why:** the deterministic validation run ID updates the same evidence on an exact rerun instead of appending duplicates. Tables 14 and 15 read this evidence before they use the parsed geometry.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING mgb_flood_area_validation_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

DECLARE OR REPLACE VARIABLE mgb_flood_area_validation_run_id STRING;
SET VAR mgb_flood_area_validation_run_id = (
    SELECT SHA2(CONCAT_WS('|', MAX(run_id), validation_rule_version), 256)
    FROM `02-silver`.silver_mgb_flood_area
);

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(
        COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Silver MGB flood-area checks failed or no evidence was persisted'
    )
)
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_mgb_flood_area'
  AND validation_run_id = mgb_flood_area_validation_run_id;

## Summary

Validation protects the grain of one MGB polygon per Bronze snapshot per parse rule version. STOP checks cover one-to-one accounting against Bronze on `object_id`, raw code fidelity, deterministic keys, complete lineage, the governed status and parse-method vocabularies, and usable geometry that rebuilds from WKB as a valid polygon in SRID 4326.

Unusable and invalid geometry, blank codes, geometry outside the degree screen, and differences from the historical reference remain visible as FLAG findings with denominators. Evidence is stored before the gate. Requires Databricks execution. No runtime result is claimed by this notebook text.

In [0]:
%sql
SELECT data_quality_attribute, status, action, failed_rows, total_rows, percentage,
       data_quality_check, details
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_mgb_flood_area'
  AND validation_run_id = mgb_flood_area_validation_run_id
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
         data_quality_attribute, data_quality_check;